In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
#import tushare as ts
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from torch.utils.data import TensorDataset
from tqdm import tqdm
import torch.nn.functional as F
import scipy.io as scio
import time 

In [2]:
import scipy
data = scipy.io.loadmat('dat.mat')
dt=data['Nt']
dx=data['Nx']
Dc=data['DataClean']
Dn=data['DataNoisey']

In [3]:
def calculate_snr(clean_data, denoised_data):
    # 确保输入是一维向量，或者直接不带 ord 参数（默认就是 Frobenius 范数，即全图能量）
    norm_clean = np.linalg.norm(clean_data)  # 默认计算所有元素的平方和开根号
    norm_diff = np.linalg.norm(clean_data - denoised_data)
    
    snr = 20 * np.log10(norm_clean / norm_diff)
    return snr

In [4]:

snr_before = calculate_snr(Dc, Dn)
print(f"去噪前信噪比 (SNR): {snr_before:.2f} dB")

去噪前信噪比 (SNR): -4.27 dB


In [5]:
def patch2d(A,l1=8,l2=8,s1=4,s2=4,mode=1):
	"""
	patch2d: decompose the image into patches:
	
	INPUT
	D: input image
	l1: first patch size
	l2: second patch size
	s1: first shifting size
	s2: second shifting size
	mode: patching mode
	
	OUTPUT
	X: patches
	
	HISTORY
	by Yangkang Chen
	Oct, 2017
	Modified on Dec 12, 2018 (the edge issue, arbitrary size for the matrix)
			 Dec 31, 2018 (tmp1=mod(n1,l1) -> tmp1=mod(n1-l1,s1))
	
	EXAMPLE 1
	
	#generate data
	from pyseistr import gensyn
	data,noisy=gensyn(noise=True);[n1,n2]=data.shape;
	import matplotlib.pyplot as plt;
	plt.subplot(1,2,1);plt.imshow(data,clim=[-0.2,0.2],aspect='auto');plt.xlabel('Trace');plt.ylabel('Time sample');
	plt.subplot(1,2,2);plt.imshow(noisy,clim=[-0.2,0.2],aspect='auto');plt.xlabel('Trace');
	plt.show();
	
	from pyseistr import patch2d
	X=patch2d(data,l1=16,l2=16,s1=8,s2=8);

	#visualize the patches
	from pyseistr import cseis
	plt.imshow(X,aspect='auto',cmap=cseis());plt.ylabel('Patch NO');plt.xlabel('Patch Pixel');plt.show()
	plt.figure(figsize=(8,8))
	for ii in range(64):
		ax=plt.subplot(8,8,ii+1)
		plt.imshow(X[3200+ii,:].reshape(16,16,order='F'),cmap=cseis(),clim=(-0.5,0.5),aspect='auto');
		plt.setp(ax.get_xticklabels(), visible=False);plt.setp(ax.get_yticklabels(), visible=False);
	plt.show()

	#reconstruct
	from pyseistr import patch2d_inv
	import numpy as np
	data2=patch2d_inv(X,n1,n2,l1=16,l2=16,s1=8,s2=8);
	print('Error=',np.linalg.norm(data.flatten()-data2.flatten()))
	
	plt.figure(figsize=(16,8));
	plt.imshow(np.concatenate([data,data2,data-data2],axis=1),aspect='auto');
	plt.show()

	EXAMPLE 2
	https://github.com/chenyk1990/mlnb/blob/main/DL_denoise_simple2D.ipynb
	
	EXAMPLE 3
	sgk_denoise() in pyseisdl/denoise.py
	"""
	[n1,n2]=A.shape;

	if mode==1: 	#possible for other patching options
		
		tmp=np.mod(n1-l1,s1);
		if tmp!=0:
			A=np.concatenate((A,np.zeros([s1-tmp,n2])),axis=0); 
		tmp=np.mod(n2-l2,s2);
		if tmp!=0:
			A=np.concatenate((A,np.zeros([A.shape[0],s2-tmp])),axis=1); 
		
		[N1,N2]=A.shape;
		X=[]
		for i1 in range(0,N1-l1+1,s1):
			for i2 in range(0,N2-l2+1,s2):
						tmp=np.reshape(A[i1:i1+l1,i2:i2+l2],(l1*l2,1),order='F');
						X.append(tmp)
		X = np.array(X)
	else:
		#not written yet
		pass;
	return X[:,:,0]



def patch2d_inv(X,n1,n2,l1=8,l2=8,s1=4,s2=4,mode=1):
	"""
	patch2d_inv: insert patches into the image
	
	INPUT
	D: input patches (sample,patchsize)
	mode: patching mode
	l1: first patch size
	l2: second patch size
	s1: first shifting size
	s2: second shifting size
	
	OUTPUT
	X: patches
	
	HISTORY
	by Yangkang Chen
	Oct, 2017
	Modified on Dec 12, 2018 (the edge issue, arbitrary size for the matrix)
				Dec 31, 2018 (tmp1=mod(n1,l1) -> tmp1=mod(n1-l1,s1))
	
	EXAMPLE 1
	
	#generate data
	from pyseistr import gensyn
	data,noisy=gensyn(noise=True);[n1,n2]=data.shape;
	import matplotlib.pyplot as plt;
	plt.subplot(1,2,1);plt.imshow(data,clim=[-0.2,0.2],aspect='auto');plt.xlabel('Trace');plt.ylabel('Time sample');
	plt.subplot(1,2,2);plt.imshow(noisy,clim=[-0.2,0.2],aspect='auto');plt.xlabel('Trace');
	plt.show();
	
	from pyseistr import patch2d
	X=patch2d(data,l1=16,l2=16,s1=8,s2=8);

	#visualize the patches
	from pyseistr import cseis
	plt.imshow(X,aspect='auto',cmap=cseis());plt.ylabel('Patch NO');plt.xlabel('Patch Pixel');plt.show()
	plt.figure(figsize=(8,8))
	for ii in range(64):
		ax=plt.subplot(8,8,ii+1)
		plt.imshow(X[3200+ii,:].reshape(16,16,order='F'),cmap=cseis(),clim=(-0.5,0.5),aspect='auto');
		plt.setp(ax.get_xticklabels(), visible=False);plt.setp(ax.get_yticklabels(), visible=False);
	plt.show()

	#reconstruct
	from pyseistr import patch2d_inv
	import numpy as np
	data2=patch2d_inv(X,n1,n2,l1=16,l2=16,s1=8,s2=8);
	print('Error=',np.linalg.norm(data.flatten()-data2.flatten()))
	
	plt.figure(figsize=(16,8));
	plt.imshow(np.concatenate([data,data2,data-data2],axis=1),aspect='auto');
	plt.show()

	EXAMPLE 2
	https://github.com/chenyk1990/mlnb/blob/main/DL_denoise_simple3D.ipynb
	
	EXAMPLE 3
	sgk_denoise() in pyseisdl/denoise.py

	"""

	if mode==1: 	#possible for other patching options

		tmp1=np.mod(n1-l1,s1);
		tmp2=np.mod(n2-l2,s2);
		if tmp1!=0 and tmp2!=0:
			A=np.zeros([n1+s1-tmp1,n2+s2-tmp2]); 
			mask=np.zeros([n1+s1-tmp1,n2+s2-tmp2]); 

		if tmp1!=0 and tmp2==0:
			A=np.zeros([n1+s1-tmp1,n2]); 
			mask=np.zeros([n1+s1-tmp1,n2]);

		if tmp1==0 and tmp2!=0:
			A=np.zeros([n1,n2+s2-tmp2]);   
			mask=np.zeros([n1,n2+s2-tmp2]);   


		if tmp1==0 and tmp2==0:
			A=np.zeros([n1,n2]); 
			mask=np.zeros([n1,n2]);

		[N1,N2]=A.shape;
		id=-1;
		for i1 in range(0,N1-l1+1,s1):
			for i2 in range(0,N2-l2+1,s2):
				id=id+1;
				A[i1:i1+l1,i2:i2+l2]=A[i1:i1+l1,i2:i2+l2]+np.reshape(X[id,:],[l1,l2],order='F');
				mask[i1:i1+l1,i2:i2+l2]=mask[i1:i1+l1,i2:i2+l2]+np.ones([l1,l2]);

		A=A/mask; 
		A=A[0:n1,0:n2];
	else:
		#not written yet
		pass;
	return A

In [28]:
import scipy
w1=16
w2=16
s1=1
s2=1
X=patch2d(Dn,w1,w2,s1,s2)
dz=1
dx=1
[nz,nx]=Dn.shape
X.shape
print(X.dtype)
X = X.astype(np.float32)
Pdata = torch.from_numpy(X)
print(Pdata.dtype)

float64
torch.float32


In [29]:
# 数据分割：80% 训练数据，20% 验证数据
train_size = int(len(Pdata) * 0.8)
train = Pdata[:train_size]
valid = Pdata[train_size:]

# 打印训练集和验证集的形状
print(f"Train shape: {train.shape}")
print(f"Valid shape: {valid.shape}")
#创建 TensorDataset
train_data = TensorDataset(train)
valid_data = TensorDataset(valid)

# 设置 batch_size
batch_size1 = 64


# 创建 DataLoader
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size1, shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_data, batch_size=batch_size1, shuffle=True)

Train shape: torch.Size([19090, 256])
Valid shape: torch.Size([4773, 256])


In [30]:
train.shape
valid.shape
print(valid.dtype)

torch.float32


In [31]:
#定义Fully connected (FC) block
class FCB(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.1):
        super().__init__()
        
        self.fc = nn.Linear(input_size, output_size)
        self.activation = nn.LeakyReLU()
        self.bn = nn.BatchNorm1d(output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc(x)
        x = self.activation(x) 
        x = self.bn(x)
        x = self.dropout(x)
        
        return x

In [32]:
class Encoder(nn.Module):
    def __init__(self, input_size, dropout=0.1):
        super().__init__()


        self.fcb0 = FCB(input_size, 128, dropout)
        self.fcb1 = FCB(128, 64, dropout)

        self.fcb2 = FCB(64, 32, dropout)

        self.fcb3 = FCB(32, 16, dropout)

        self.fcb4 = FCB(16, 8, dropout)

        self.fcb5 = FCB(8, 4, dropout)

        self.fcb6 = FCB(4, 4, dropout)

        


    def forward(self, x):

        #x = x.reshape(x.shape[0],1,x.shape[1])

        x1 = self.fcb0(x)#x->128
        x2 = self.fcb1(x1)

        x3 = self.fcb2(x2)

        x4 = self.fcb3(x3)

        x5 = self.fcb4(x4)

        x6 = self.fcb5(x5)

        x7 = self.fcb6(x6)

        x8 = self.fcb6(x7)
        

        

        return x2,x3,x4,x5,x6,x7,x8

In [33]:
class Decoder(nn.Module):
    def __init__(self, output_size, dropout=0.1):
        super().__init__()
        self.pab1 = FCB(8, 8, dropout)
        self.pab2 = FCB(16, 16, dropout)
        self.pab3 = FCB(32, 32, dropout)
        self.pab4 = FCB(64, 64, dropout)
        self.pab5 = FCB(128, 128, dropout)
        self.pab6 = FCB(128, output_size, dropout)

        self.activation = nn.Identity()


    def forward(self,x2,x3,x4,x5,x6,x7,x8):
        
        x = torch.cat((x6,x8),dim=1)
        x = self.pab1(x)
        x = torch.cat((x,x5),dim=1)
        x = self.pab2(x)
        x = torch.cat((x,x4),dim=1)
        x = self.pab3(x)
        x = torch.cat((x,x3),dim=1)
        x = self.pab4(x)
        x = torch.cat((x,x2),dim=1)
        x = self.pab5(x)
        x = self.pab6(x)

        x = self.activation(x)
        return x

In [34]:
class AutoEncoder(nn.Module):
    def __init__(self,input_size, output_size):
        super().__init__()
        self.encoder = Encoder(input_size)
        self.decoder = Decoder(output_size)

    def forward(self, x):
        x2,x3,x4,x5,x6,x7,x8= self.encoder(x)
        x = self.decoder(x2,x3,x4,x5,x6,x7,x8)

        return x

In [35]:
device = torch.device("cuda") 
model = AutoEncoder(w1*w2,w1*w2).to(device)
#将模型转移到GPU上
#criterion = MeanHuberLoss(delta=1.3)
#criterion = WelschLoss(delta=0.5)
#criterion = Loss0(delta=0.46,r=0.05)#0.5 and 0.2,SNR:-8dB ok
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.001)

In [36]:
start_time = time.time()
es_cnt = 0
es_thres = 5
prev_train_loss = float('inf')
loss_train = []
loss_valid = []
num_epochs = 100 # 总训练轮数
#num_batch_train = 0
for epoch in range(num_epochs):
  #train_bar = tqdm(train_loader)
  train_loss = 0.0
  
  for i , (batch) in enumerate(train_loader):

    # 数据转到device
    train_batch = batch[0].to(device)
    
    # 训练步骤  
    optimizer.zero_grad()
    outputs = model(train_batch)
    loss = criterion(outputs, train_batch)
    loss.backward() 
    optimizer.step()
    
    train_loss += loss.item()
    #num_batch_train +=1
  #train_loss除以所有bacth个数
  train_loss = train_loss/(np.ceil(train.size(0)/batch_size1))
  loss_train.append(train_loss)
    



  # 验证
  valid_loss = 0.0
  #num_batch_vaild = 0
  with torch.no_grad():
    for i , (batch) in enumerate(valid_loader):
    #for batch in vaild_loader:
    
      val_batch = batch[0].to(device)
      
      outputs = model(val_batch)
      loss = criterion(outputs, val_batch)
      valid_loss += loss.item()
      #num_batch_vaild += 1
    valid_loss = valid_loss/(np.ceil(valid.size(0)/batch_size1))
    loss_valid.append(valid_loss)
    print("Epoch [{}/{}], Train Loss: {:.4f}, Valid Loss: {:.4f}".format(epoch+1, num_epochs, train_loss, valid_loss))

    
    # Early stopping
    if train_loss - prev_train_loss >= 0:
        es_cnt += 1
    else:
        #es_cnt = 0
        pass

    if es_cnt >= es_thres:
        print(f"Early stopped at epoch {epoch}, train loss stop improving")
        break  

    prev_train_loss = train_loss        
print("Training finished")
current_time = time.time()
time_sum = current_time-start_time
print(time_sum)

Epoch [1/100], Train Loss: 0.7850, Valid Loss: 0.6153
Epoch [2/100], Train Loss: 0.3625, Valid Loss: 0.3248
Epoch [3/100], Train Loss: 0.1835, Valid Loss: 0.1782
Epoch [4/100], Train Loss: 0.1084, Valid Loss: 0.1126
Epoch [5/100], Train Loss: 0.0824, Valid Loss: 0.0864
Epoch [6/100], Train Loss: 0.0750, Valid Loss: 0.0767
Epoch [7/100], Train Loss: 0.0730, Valid Loss: 0.0731
Epoch [8/100], Train Loss: 0.0720, Valid Loss: 0.0718
Epoch [9/100], Train Loss: 0.0712, Valid Loss: 0.0712
Epoch [10/100], Train Loss: 0.0704, Valid Loss: 0.0706
Epoch [11/100], Train Loss: 0.0696, Valid Loss: 0.0700
Epoch [12/100], Train Loss: 0.0689, Valid Loss: 0.0692
Epoch [13/100], Train Loss: 0.0681, Valid Loss: 0.0683
Epoch [14/100], Train Loss: 0.0673, Valid Loss: 0.0675
Epoch [15/100], Train Loss: 0.0666, Valid Loss: 0.0666
Epoch [16/100], Train Loss: 0.0660, Valid Loss: 0.0659
Epoch [17/100], Train Loss: 0.0654, Valid Loss: 0.0653
Epoch [18/100], Train Loss: 0.0648, Valid Loss: 0.0650
Epoch [19/100], Tra

In [18]:
loss_train = pd.DataFrame(loss_train)
loss_valid = pd.DataFrame(loss_valid)

loss = pd.concat([loss_train,loss_valid],axis=1)

In [19]:
loss.columns = ['train_loss','vaild_loss']

In [20]:
torch.save(model.state_dict(), r'.\model_2dsyn1patch4.pth')

In [21]:
model = AutoEncoder(w1*w2,w1*w2).to(device)
Pdata = Pdata.to(device)
model.load_state_dict(torch.load(r'.\model_2dsyn1patch4.pth'))
model.eval()
with torch.no_grad():
    output = model(Pdata)
    loss = criterion(output, Pdata)
output = output.cpu()
output = output.numpy()
# ========== 新增：重构去噪后的数据并计算信噪比 ==========
# 重构去噪后的数据
n1 = Dn.shape[0]  # 对应MATLAB：n1 = size(Dn, 1)
n2 = Dn.shape[1]  # 对应MATLAB：n2 = size(Dn, 2)
D_denoised = patch2d_inv(output,n1,n2,w1,w2,s1,s2)
scipy.io.savemat(r"output_syn2d4.mat", 
        {'D_denoised': D_denoised})  # 'D_denoised'是MAT文件里的变量名
# 计算去噪前的信噪比（原始含噪数据 vs 干净数据）
snr_before = calculate_snr(Dc, Dn)
#计算去噪后的信噪比（去噪后数据 vs 干净数据）
snr_after = calculate_snr(Dc, D_denoised)

#输出信噪比结果
print("="*50)
print(f"去噪前信噪比 (SNR): {snr_before:.2f} dB")
print(f"去噪后信噪比 (SNR): {snr_after:.2f} dB")
print(f"信噪比提升: {snr_after - snr_before:.2f} dB")
print("="*50)

去噪前信噪比 (SNR): -4.27 dB
去噪后信噪比 (SNR): -1.84 dB
信噪比提升: 2.43 dB


In [1]:
data = scipy.io.loadmat('output_syn2d.mat')
D_denoised=data['D_denoised']

NameError: name 'scipy' is not defined

In [30]:
#48*48
# 计算去噪前的信噪比（原始含噪数据 vs 干净数据）
snr_before = calculate_snr(Dc, Dn)
#计算去噪后的信噪比（去噪后数据 vs 干净数据）
snr_after = calculate_snr(Dc, D_denoised)

#输出信噪比结果
print("="*50)
print(f"去噪前信噪比 (SNR): {snr_before:.2f} dB")
print(f"去噪后信噪比 (SNR): {snr_after:.2f} dB")
print(f"信噪比提升: {snr_after - snr_before:.2f} dB")
print("="*50)

去噪前信噪比 (SNR): -4.27 dB
去噪后信噪比 (SNR): 2.50 dB
信噪比提升: 6.77 dB


In [22]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()  # 清理进程间显存缓存，进一步减少碎片

In [ ]:
#loss.to_csv(r'loss_2dsyn1patch.csv',index=False)

In [38]:
#loss_0 = []
#start_time = time.time()
#es_cnt = 0
#es_thres = 5
#prev_train_loss = float('inf')
#loss_train = []
#loss_vaild = []
#num_epochs = 100 # 总训练轮数
##num_batch_train = 0
#for epoch in range(num_epochs):
#  #train_bar = tqdm(train_loader)
#  train_loss = 0.0
#  
#  for i , (batch) in enumerate(train_loader):
#
#    # 数据转到device
#    train_batch = batch[0].to(device)
#    
#    # 训练步骤  
#    optimizer.zero_grad()
#    outputs = model(train_batch)
#    loss = criterion(outputs, train_batch)
#    loss.backward() 
#    optimizer.step()
#
#    loss_0.append(loss.item())
#
#    train_loss += loss.item()
#    #num_batch_train +=1
#  #train_loss除以所有bacth个数
#  train_loss = train_loss/(np.ceil(train.size(0)/batch_size1))
#  loss_train.append(train_loss)
#    
#
#
#
#  # 验证
#  valid_loss = 0.0
#  #num_batch_vaild = 0
#  with torch.no_grad():
#    for i , (batch) in enumerate(vaild_loader):
#    #for batch in vaild_loader:
#    
#      val_batch = batch[0].to(device)
#      
#      outputs = model(val_batch)
#      loss = criterion(outputs, val_batch)
#      valid_loss += loss.item()
#      #num_batch_vaild += 1
#    valid_loss = valid_loss/(np.ceil(vaild.size(0)/batch_size1))
#    loss_vaild.append(valid_loss)
#    print("Epoch [{}/{}], Train Loss: {:.4f}, Valid Loss: {:.4f}".format(epoch+1, num_epochs, train_loss, valid_loss))
#
#    
#    # Early stopping
#    if train_loss - prev_train_loss >= 0:
#        es_cnt += 1
#    else:
#        #es_cnt = 0
#        pass
#
#    if es_cnt >= es_thres:
#        print(f"Early stopped at epoch {epoch}, train loss stop improving")
#        break  
#    
#
#
##   if epoch == 10:
##     torch.save(model.state_dict(), r'.\model_epochs10.pth')
##   elif epoch == 20:
##     torch.save(model.state_dict(), r'.\model_epochs20.pth')
##   elif epoch == 30:
##     torch.save(model.state_dict(), r'.\model_epochs30.pth')
##   elif epoch == 40:
##     torch.save(model.state_dict(), r'.\model_epochs40.pth')
##   elif epoch == 50:
##     torch.save(model.state_dict(), r'.\model_epochs50.pth')
##   elif epoch == 60:
##     torch.save(model.state_dict(), r'.\model_epochs60.pth')
##   elif epoch == 70:
##     torch.save(model.state_dict(), r'.\model_epochs70.pth')
##   elif epoch == 80:
##     torch.save(model.state_dict(), r'.\model_epochs80.pth')
##   elif epoch == 90:
##     torch.save(model.state_dict(), r'.\model_epochs90.pth')
##   elif epoch == 100:
##     torch.save(model.state_dict(), r'.\model_epochs100.pth')
##   else:
##       pass
#
#    prev_train_loss = train_loss
#  print('loss_train: ', loss_train)
#  print('vaild_train: ',loss_vaild)          
#print("Training finished")
#current_time = time.time()
#time_sum = current_time-start_time
#print(time_sum)